In [ ]:

import pandas as pd
import os

file_path = "/kaggle/input/datasets/prachishinde18/bank-dataset/Bank Data Analystics Dashboard.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Rows and Columns:", df.shape)

display(df.head())


In [ ]:

import pandas as pd
from IPython.display import display, HTML

df.columns = df.columns.str.strip()

def find_col(*names):
    for name in names:
        for col in df.columns:
            if name.strip().lower() in [c.strip().lower() for c in df.columns]:
                return next(c for c in df.columns if c.strip().lower() == name.strip().lower())
    return None

funded_col = find_col("Funded Amount")
client_col = find_col("Client id")
principal_col = find_col("Total Rec Prncp")

# Convert amount columns to numeric
for col in [funded_col, principal_col]:
    if col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate KPIs
total_funded = df[funded_col].sum() if funded_col else None
total_clients = df[client_col].nunique() if client_col else None
principal_received = df[principal_col].sum() if principal_col else None
avg_loan = df[funded_col].mean() if funded_col else None

kpis = [
    ("TOTAL FUNDED AMOUNT",
     f"₹{total_funded/10000000:,.2f} Cr" if total_funded is not None else "Column not found"),
    ("TOTAL CLIENTS",
     f"{total_clients:,}" if total_clients is not None else "Column not found"),
    ("PRINCIPAL RECEIVED",
     f"₹{principal_received/10000000:,.2f} Cr" if principal_received is not None else "Column not found"),
    ("AVERAGE LOAN AMOUNT",
     f"₹{avg_loan/1000:,.2f}K" if avg_loan is not None else "Column not found")
]

cards = ""
for title, value in kpis:
    cards += f"""
    <div style="background:#123456;color:white;padding:22px;
    margin:8px;border-radius:12px;min-width:200px;flex:1;text-align:center;">
        <h4 style="color:#C8E6FF;">{title}</h4>
        <h2>{value}</h2>
    </div>
    """

display(HTML(
    f'<div style="display:flex;flex-wrap:wrap;">{cards}</div>'
))


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

df.columns = df.columns.str.strip()

df["Funded Amount"] = pd.to_numeric(
    df["Funded Amount"], errors="coerce"
)

branch_data = (
    df.groupby("Branch Name")["Funded Amount"]
    .sum()
    .nlargest(10) / 10000000
)

plt.figure(figsize=(10, 6))
branch_data.sort_values().plot(kind="barh")
plt.title("Top 10 Branches by Loan Amount")
plt.xlabel("Loan Amount (Crore)")
plt.ylabel("Branch Name")
plt.tight_layout()
plt.show()


In [ ]:

state_data = (
    df.groupby("State Name")["Funded Amount"]
    .sum()
    .nlargest(5) / 10000000
)

plt.figure(figsize=(9, 5))
state_data.sort_values().plot(kind="barh")
plt.title("Top 5 States by Loan Amount")
plt.xlabel("Loan Amount (Crore)")
plt.ylabel("State Name")
plt.tight_layout()
plt.show()


In [ ]:

status_col = "Loan Status2" if "Loan Status2" in df.columns else "Loan Status"

status_data = (
    df.groupby(status_col)["Funded Amount"]
    .sum()
    .sort_values(ascending=False) / 10000000
)

plt.figure(figsize=(10, 5))
status_data.plot(kind="bar")
plt.title("Loan Status Analysis")
plt.xlabel("Loan Status")
plt.ylabel("Funded Amount (Crore)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

grade_col = "Grrade" if "Grrade" in df.columns else "Grade"

df["Loan Amount"] = pd.to_numeric(
    df["Loan Amount"], errors="coerce"
)

grade_data = (
    df[df[grade_col].notna()]
    .groupby(grade_col)["Loan Amount"]
    .mean()
    .sort_index()
)

plt.figure(figsize=(9, 5))
grade_data.plot(kind="bar")
plt.title("Average Loan Amount by Grade")
plt.xlabel("Loan Grade")
plt.ylabel("Average Loan Amount (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

principal_col = "Total Rec Prncp"
interest_col = (
    "Total Rrec int"
    if "Total Rrec int" in df.columns
    else "Total Rec int"
)

df[principal_col] = pd.to_numeric(
    df[principal_col], errors="coerce"
)
df[interest_col] = pd.to_numeric(
    df[interest_col], errors="coerce"
)

amounts = [
    df[principal_col].sum() / 10000000,
    df[interest_col].sum() / 10000000
]

plt.figure(figsize=(8, 5))
plt.bar(
    ["Principal Received", "Interest Received"],
    amounts
)
plt.title("Principal vs Interest Analysis")
plt.ylabel("Amount (Crore)")
plt.tight_layout()
plt.show()
